# Feature Selection
Follow-up to `02_exploratory_analysis.ipynb` and `02_1_correlation_tests.ipynb`. Those notebooks showed which columns relate to `route_status`. This notebook decides which columns the model is actually allowed to use, and saves a smaller, model-ready file for `03_baseline_model.ipynb`.

Steps:
1. Set the train/test split, so every decision below is made on training rows only
2. Remove the column that gives away the answer
3. Remove columns that only identify the route or the week
4. Remove columns that are copies of other columns
5. Test whether each remaining feature (and the optional event group) actually helps a model
6. Save `data/preprocessed/model_ready.csv`

`data/preprocessed/supply_chain_disruption.csv` from 01 is **not** changed. It stays the full record of the merge, and this notebook only selects from it.

## 1. Load Merged Data
Load `supply_chain_disruption.csv` and convert `date` back to a real date type (01 saved it as `MM/DD/YYYY` text).

Rows are sorted by route number, then date. Sorting `route_id` as text would put `R10` before `R2`, so the number is pulled out of the ID for sorting.

In [1]:
import os

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

merged_df = pd.read_csv("../data/preprocessed/supply_chain_disruption.csv")
merged_df["date"] = pd.to_datetime(merged_df["date"], format="%m/%d/%Y")

# Sort by route number (R1, R2, ..., R50), then by date
route_number = merged_df["route_id"].str.lstrip("R").astype(int)
merged_df = (
    merged_df.assign(route_number=route_number)
    .sort_values(["route_number", "date"])
    .drop(columns="route_number")
    .reset_index(drop=True)
)

print("Shape:", merged_df.shape)

Shape: (31300, 48)


## 2. Set the Train/Test Split
Rows are split **randomly**: 80% train, 20% test.

Why random:
- Each row (one route in one week) is treated as an independent data point. 02_1 supports this: last week's numbers and last week's status have almost no correlation with this week's status (about 0.02), so knowing one week tells you almost nothing about the next.
- The goal is to classify a route-week from its own conditions, not to forecast a time series, so the split doesn't need to keep weeks in order.

Why **stratified** (`stratify=route_status`): Disrupted is only about 3% of rows. A plain random split could put noticeably more or fewer Disrupted rows in the test set just by chance. Stratifying keeps the Normal / Delayed / Disrupted mix the same in both sets.

Why 80/20: the test set gets about 200 Disrupted rows, enough for a stable Disrupted score. A 10% test set would have only about 100.

**Rule for the rest of this notebook:** every check and every decision uses training rows only. The test rows are only counted here and saved at the end. If we picked features because they scored well on the test set, the final test score in 03 would no longer be a fair estimate.

Notes:
- The cross-validation folds are **not** saved here. Folds are part of evaluating a model, so 03 creates its own (with a fixed `random_state`, so they are reproducible).
- 01 found the last 30 weeks (from 2026-06-07) have many more Disrupted rows (11.9% in 2026 vs. about 2-3% in earlier years). With a random split, those weeks are spread across both sets. `date` is kept in the saved file, so 03 can still report the test score for those weeks separately as an extra check.

In [2]:
from sklearn.model_selection import train_test_split

# Random 80/20 split that keeps the same status mix in both sets
train_index, test_index = train_test_split(
    merged_df.index, test_size=0.2, stratify=merged_df["route_status"], random_state=42
)
merged_df["split"] = "train"
merged_df.loc[test_index, "split"] = "test"

status_order = ["Normal", "Delayed", "Disrupted"]
split_counts = pd.crosstab(merged_df["split"], merged_df["route_status"])[status_order]
split_percent = (pd.crosstab(merged_df["split"], merged_df["route_status"], normalize="index") * 100).round(1)[status_order]

pd.concat({"count": split_counts, "percent": split_percent}, axis=1)

count                   percent                  
route_status Normal Delayed Disrupted  Normal Delayed Disrupted
split                                                          
test           2053    4008       199    32.8    64.0       3.2
train          8211   16030       799    32.8    64.0       3.2

Both sets have the same status mix (about 33% Normal, 64% Delayed, 3% Disrupted), so scores on the test set will be comparable to scores in cross-validation.

From here on, `train_df` is the only data used for decisions.

In [3]:
train_df = merged_df[merged_df["split"] == "train"].copy()
print("Training rows:", len(train_df))

Training rows: 25040


## 3. Remove the Column That Gives Away the Answer
02_1 showed that `route_status` is decided **only** by `shipping_delay_days`:
- under 5 days -> Normal
- 5 to 12 days -> Delayed
- 12 or more days -> Disrupted

A model given this column would just learn the two cutoffs. The delay also isn't known until the shipment is already late, so it can't be used to warn about anything. Quick re-check on the training rows:

In [4]:
train_df.groupby("route_status")["shipping_delay_days"].agg(["min", "max"]).loc[status_order]

,min,max
route_status,,
Normal,0.0,4.995
Delayed,5.0,11.990
Disrupted,12.0,20.952


In [5]:
# Never a model input, and not saved to model_ready.csv
target_leak_columns = ["shipping_delay_days"]

## 4. Remove Columns That Only Identify the Route
Many columns come from `trade_routes.csv` and `country_metadata.csv`. They describe the route or its countries, so they have the **same value in every week of a route**. With only 50 routes, these columns can only ever take 50 different combinations, which makes them a long way of writing `route_id`.

The check below counts, for each column, the largest number of different values found inside a single route. A result of `1` means the column never changes within a route.

In [6]:
# Largest number of different values any single route has, for each column
values_per_route = train_df.drop(columns=["route_id", "split"]).groupby(train_df["route_id"]).nunique().max()

route_level_columns = values_per_route[values_per_route == 1].index.tolist()
print(f"{len(route_level_columns)} columns never change within a route:")
for column in route_level_columns:
    print(" ", column)

24 columns never change within a route:
  origin_country
  destination_country
  shipping_method
  trade_route_type
  distance_km
  estimated_transit_days
  origin_region
  origin_event_region
  origin_income_group
  origin_port_capacity_index
  origin_logistics_performance_index
  origin_trade_dependency_score
  origin_gdp_per_capita
  origin_population
  origin_iso3
  destination_region
  destination_event_region
  destination_income_group
  destination_port_capacity_index
  destination_logistics_performance_index
  destination_trade_dependency_score
  destination_gdp_per_capita
  destination_population
  destination_iso3


Why these are dropped:
- **No signal:** 02_1 showed the percent of Normal / Delayed / Disrupted is almost the same for every country, shipping method and route type (about 33% / 64% / 3% everywhere). The differences between routes are about the size of random noise.
- **Memorizing routes:** a tree model can use these columns to learn "route 17 had a few bad weeks" instead of learning what causes disruptions. That doesn't carry over to new conditions.
- **Simpler model:** this removes half of the 48 columns, which makes the model and its explanations much easier to read.

The details aren't lost. They are still in `trade_routes.csv` and `country_metadata.csv`, and `route_id` is kept in the saved file, so they can be joined back for dashboards (e.g. "risk by route" or "risk by origin country"). They just shouldn't be model inputs.

## 5. Remove Columns That Only Identify the Week
The opposite problem: some columns are the **same for every route in a given week**. The market prices come from `commodity_market.csv`, which has one row per week, so all 50 routes get the same prices.

The check below counts the largest number of different values found inside a single week. A result of `1` means every route has the same value that week.

In [7]:
# Largest number of different values any single week has, for each column
values_per_week = train_df.drop(columns=["date", "split"]).groupby(train_df["date"]).nunique().max()
week_level_columns = values_per_week[values_per_week == 1].index.tolist()

print("Same for every route in a week:")
for column in week_level_columns:
    print(" ", column)

Same for every route in a week:
  covid_supply_shock_active
  events_warmup
  fuel_cost_index
  commodity_price_index
  natural_gas_price
  steel_price
  wheat_price
  copper_price


Three kinds of columns show up:
- **Market prices** (`fuel_cost_index`, `commodity_price_index`, `natural_gas_price`, `steel_price`, `wheat_price`, `copper_price`): dropped. Each has only one value per week, so to a tree model they work like a calendar. In cross-validation, the same week appears in both the training and the validation folds (50 routes share it), so the model can recognize the week instead of learning a real cause. That makes them look better in cross-validation than they really are. 02_1 also found their correlation with status is close to 0.
- **`events_warmup`**: dropped. It is a note from 01 saying "the event counts for the first 52 weeks may be too low", not a feature. If the event features are used later, the modeling step can drop those 52 weeks instead.
- **`covid_supply_shock_active`**: not dropped here. It is the same for every route because COVID was global, but it marks a real, known event rather than just a date. It is tested with the other event features in section 8.

In [8]:
market_columns = [
    "fuel_cost_index", "commodity_price_index",
    "natural_gas_price", "steel_price", "wheat_price", "copper_price",
]
bookkeeping_columns = ["events_warmup"]

## 6. Remove Columns That Are Copies of Other Columns
02_1 found some pairs with a correlation above 0.8. Here we check how closely each column can be **rebuilt** from other columns. If it can be rebuilt almost exactly, it adds no new information. It only splits the credit between two columns, which makes feature importance harder to explain.

We use R²: the share of a column's variation that a simple linear formula of the other columns explains. 1.0 means a perfect copy, 0 means unrelated.

**Each check is done separately for two periods:** before 2026-06-07, and from 2026-06-07 on. 01 found that the commodity values change scale in exactly those last 30 weeks, and the formulas linking some columns change there too. One formula fitted across both periods would look imperfect even for a column that is an exact copy within each period. This is only used to understand the columns; the date is not a model feature.

In [9]:
# The two periods 01 found (commodity values change scale from 2026-06-07)
period = np.where(train_df["date"] >= pd.Timestamp("2026-06-07"), "from 2026-06-07", "before 2026-06-07")
period_names = ["before 2026-06-07", "from 2026-06-07"]

def copy_score(target, predictors):
    """R² of a linear formula that rebuilds `target` from the `predictors` columns, fitted separately in each period."""
    scores = {}
    for name in period_names:
        rows = period == name
        X, y = predictors[rows], target[rows]
        scores[f"R² {name}"] = LinearRegression().fit(X, y).score(X, y)
    return scores

# Shipping method as 0/1 columns, needed for the carbon check
shipping_method_dummies = pd.get_dummies(train_df["shipping_method"], dtype=float)

copy_checks = pd.DataFrame([
    {"column": "container_availability_index", "rebuilt from": "geopolitical_risk_score",
     **copy_score(train_df["container_availability_index"], train_df[["geopolitical_risk_score"]])},
    {"column": "freight_cost_usd", "rebuilt from": "distance_km, fuel_cost_index, geopolitical_risk_score",
     **copy_score(train_df["freight_cost_usd"], train_df[["distance_km", "fuel_cost_index", "geopolitical_risk_score"]])},
    # Carbon is a product (volume x distance x factor), so it is checked on the log scale, where products become sums
    {"column": "log(carbon_emissions_tonnes)", "rebuilt from": "log(trade_volume_tonnes), log(distance_km), shipping_method",
     **copy_score(np.log(train_df["carbon_emissions_tonnes"]),
                  pd.concat([np.log(train_df[["trade_volume_tonnes", "distance_km"]]), shipping_method_dummies], axis=1))},
    {"column": "origin_active_event_risk", "rebuilt from": "origin_active_event_count",
     **copy_score(train_df["origin_active_event_risk"], train_df[["origin_active_event_count"]])},
    {"column": "destination_active_event_risk", "rebuilt from": "destination_active_event_count",
     **copy_score(train_df["destination_active_event_risk"], train_df[["destination_active_event_count"]])},
])

copy_checks.round(3)

,column,rebuilt from,R² before 2026-06-07,R² from 2026-06-07
0,container_availability_index,geopolitical_risk_score,1.000,0.966
1,freight_cost_usd,"distance_km, fuel_cost_index, geopolitical_ris...",0.999,0.999
2,log(carbon_emissions_tonnes),"log(trade_volume_tonnes), log(distance_km), sh...",1.000,1.000
3,origin_active_event_risk,origin_active_event_count,0.872,0.886
4,destination_active_event_risk,destination_active_event_count,0.868,0.859


### Reading the Copies
- **`container_availability_index`** is a perfect copy of `geopolitical_risk_score` before 2026-06-07. From 2026-06-07 the link loosens: geopolitical risk jumps in those weeks while container availability doesn't. So outside of "this row is from the last 30 weeks", it carries no new information (section 8 tests exactly this). We keep `geopolitical_risk_score`: it is the underlying driver (risk goes up, availability goes down), so it gives the clearer explanation. This is why the two were #1 and #2 in 02_1: it was mostly the same signal counted twice.
- **`carbon_emissions_tonnes`** is exactly `trade_volume_tonnes` x `distance_km` x a fixed factor for each shipping method, in both periods. It is a calculated column, so it is dropped and `trade_volume_tonnes` is kept.
- **`freight_cost_usd`** is almost fully explained by distance, fuel price and geopolitical risk (R² 0.999 in both periods). But "almost" matters here. See the next check.
- **Event count vs. event risk** are close, but not exact copies. We keep the **risk** columns because they weight each event by how serious it is (`risk_increase`), instead of counting a small strike the same as a war. The count columns are dropped.

### Freight Cost Hides the Delay
The small part of `freight_cost_usd` that distance, fuel and risk **don't** explain could just be noise. If it is noise, it shouldn't relate to anything. The check below takes that leftover part and compares it with `shipping_delay_days`, then refits the formula with the delay added.

In [10]:
freight_formula_columns = ["distance_km", "fuel_cost_index", "geopolitical_risk_score"]

freight_checks = {}
for name in period_names:
    rows = train_df[period == name]
    freight_formula = LinearRegression().fit(rows[freight_formula_columns], rows["freight_cost_usd"])

    # The part of freight cost the formula can't explain
    freight_leftover = rows["freight_cost_usd"] - freight_formula.predict(rows[freight_formula_columns])

    freight_checks[name] = {
        "leftover vs. delay (correlation)": freight_leftover.corr(rows["shipping_delay_days"]),
        "R² without delay": copy_score(train_df["freight_cost_usd"], train_df[freight_formula_columns])[f"R² {name}"],
        "R² with delay added": copy_score(train_df["freight_cost_usd"], train_df[freight_formula_columns + ["shipping_delay_days"]])[f"R² {name}"],
    }

pd.DataFrame(freight_checks).round(4)

,before 2026-06-07,from 2026-06-07
leftover vs. delay (correlation),0.8684,0.8715
R² without delay,0.9989,0.9990
R² with delay added,1.0000,1.0000


In both periods, the leftover part is strongly tied to the delay, and adding the delay rebuilds freight cost **exactly**. So freight cost was calculated partly **from the delay**, which means it contains a piece of the answer, just like `shipping_delay_days` itself. A model would use that piece to get the status right, and it would look better than it really is.

`freight_cost_usd` is treated as a **second target leak**, not just a copy. This also makes sense in real life: the final freight bill for a week isn't known until the shipment has arrived.

In [11]:
# freight_cost_usd contains part of shipping_delay_days, so it is a leak, not just a copy
target_leak_columns.append("freight_cost_usd")

copy_columns = [
    "container_availability_index",     # copy of geopolitical_risk_score
    "carbon_emissions_tonnes",          # calculated from trade volume, distance and shipping method
    "origin_active_event_count",        # risk columns kept instead (weighted by severity)
    "destination_active_event_count",
]

## 7. Feature Groups
Every column now has a place. The columns that are left are split into:
- **`core_features`**: the 4 columns that are left from 02 and 02_1's list of columns linked to status (after removing the leak and the copies). They change every week on every route. These are the candidates; section 8 checks that each one is really needed.
- **`optional_groups`**: the geopolitical event features from 01. 02_1 found their correlation with status is close to 0, but they are the project's main "early warning" candidates, so they are tested in section 8 instead of dropped on sight.

Like the column ordering in 01, the code checks that **every column is placed exactly once**. If a new column is added to the merged file later and not listed here, it raises an error instead of being silently dropped or silently used.

In [12]:
id_columns = ["date", "route_id", "split"]
target_column = "route_status"

core_features = [
    "geopolitical_risk_score",     # higher risk -> worse status (strongest single signal)
    "port_congestion_index",       # higher congestion -> worse status
    "weather_disruption_score",    # worse weather -> worse status
    "trade_volume_tonnes",         # lower volume goes with worse status
]

optional_groups = {
    "events": [
        "origin_active_event_risk", "destination_active_event_risk",
        "covid_supply_shock_active", "russia_ukraine_war_active",
    ],
}

dropped_columns = {
    "target leak": target_leak_columns,
    "route-level": route_level_columns,
    "market (week-level)": market_columns,
    "bookkeeping": bookkeeping_columns,
    "copies": copy_columns,
}

placed_columns = (
    id_columns + [target_column] + core_features
    + [c for group in optional_groups.values() for c in group]
    + [c for group in dropped_columns.values() for c in group]
)

# Every column must be placed exactly once
missing_columns = set(merged_df.columns) - set(placed_columns)
unknown_columns = set(placed_columns) - set(merged_df.columns)
assert len(placed_columns) == len(set(placed_columns)), "A column is placed twice"
assert not missing_columns, f"Columns not placed anywhere: {missing_columns}"
assert not unknown_columns, f"Listed columns that don't exist: {unknown_columns}"

print(f"core ({len(core_features)}): {', '.join(core_features)}")
for group, columns in optional_groups.items():
    print(f"optional '{group}' ({len(columns)}): {', '.join(columns)}")
for reason, columns in dropped_columns.items():
    print(f"dropped, {reason} ({len(columns)}): {', '.join(columns)}")

core (4): geopolitical_risk_score, port_congestion_index, weather_disruption_score, trade_volume_tonnes
optional 'events' (4): origin_active_event_risk, destination_active_event_risk, covid_supply_shock_active, russia_ukraine_war_active
dropped, target leak (2): shipping_delay_days, freight_cost_usd
dropped, route-level (24): origin_country, destination_country, shipping_method, trade_route_type, distance_km, estimated_transit_days, origin_region, origin_event_region, origin_income_group, origin_port_capacity_index, origin_logistics_performance_index, origin_trade_dependency_score, origin_gdp_per_capita, origin_population, origin_iso3, destination_region, destination_event_region, destination_income_group, destination_port_capacity_index, destination_logistics_performance_index, destination_trade_dependency_score, destination_gdp_per_capita, destination_population, destination_iso3
dropped, market (week-level) (6): fuel_cost_index, commodity_price_index, natural_gas_price, steel_price,

## 8. Do the Features Earn Their Place?
Correlation only measures straight-line relationships, so this section asks a model directly: **does adding or removing a column change how well it predicts?**

**Setup:**
- **Model:** scikit-learn's `HistGradientBoostingClassifier`, a gradient-boosted tree model similar to LightGBM and XGBoost. It is only used to compare feature sets; choosing and tuning the real model is 03's job.
- **Class weights** (`class_weight="balanced"`): each Disrupted row counts more during training, so the model can't ignore the ~3% minority. No resampling.
- **Two scores:**
  - *macro-F1*: the average F1 of the 3 statuses, so Disrupted counts as much as Delayed
  - *Disrupted PR-AUC*: how well the model ranks Disrupted rows above the rest, without depending on a decision threshold. Random guessing scores about 0.032 (the Disrupted rate).

**Two checks, both on training rows only:**
1. **Stratified 5-fold cross-validation.** Each fold keeps the same Normal / Delayed / Disrupted mix as the full training set, so every fold has enough Disrupted rows to score. The same 5 folds are used for every feature set, so differences come from the features, not from which rows landed in which fold.
2. **Unseen-weeks check.** Also 5 stratified folds, but every week is kept whole: all routes of a week land in the same fold, so the model is always scored on weeks it has never seen. In plain cross-validation, the other routes from the same week sit in the training folds, so a week-level column (like a market price) can help just by recognizing the week. This check removes the most direct version of that shortcut. It still uses only training rows; the test set stays untouched.

**Decision rule** (the `verdict` column), comparing each feature set with the baseline set:
- **better**: cross-validation macro-F1 goes up by more than its noise (twice its standard error, std / √5), the unseen-weeks macro-F1 also goes up, **and** Disrupted PR-AUC doesn't drop in either check. Disrupted is the status we most care about, so a feature set that helps the big classes but hurts Disrupted isn't counted as better.
- **worse**: cross-validation macro-F1 goes down by more than its noise, and the unseen-weeks macro-F1 also goes down.
- **no difference**: anything else. In that case the simpler feature set wins.

In [13]:
y_train = train_df[target_column]
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train_df, y_train))

# Unseen-weeks check: same idea, but all routes of a week stay together in one fold
week_folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(train_df, y_train, groups=train_df["date"]))

# Confirm no week appears in both the training and the validation part of any fold
for train_index, valid_index in week_folds:
    assert not set(train_df["date"].iloc[train_index]) & set(train_df["date"].iloc[valid_index])
print("Weeks per validation fold:", [train_df["date"].iloc[valid_index].nunique() for _, valid_index in week_folds])

def new_model():
    return HistGradientBoostingClassifier(class_weight="balanced", learning_rate=0.05, max_iter=300, random_state=42)

def score(model, X, y):
    """Macro-F1 and Disrupted PR-AUC of a fitted model."""
    disrupted_probability = model.predict_proba(X)[:, list(model.classes_).index("Disrupted")]
    return {
        "macro_f1": f1_score(y, model.predict(X), average="macro"),
        "disrupted_pr_auc": average_precision_score(y == "Disrupted", disrupted_probability),
    }

def evaluate(feature_columns):
    """Scores on each of the 5 cross-validation folds, and the average over the 5 unseen-weeks folds."""
    def run_folds(fold_list):
        scores = []
        for train_index, valid_index in fold_list:
            model = new_model().fit(train_df.iloc[train_index][feature_columns], y_train.iloc[train_index])
            scores.append(score(model, train_df.iloc[valid_index][feature_columns], y_train.iloc[valid_index]))
        return pd.DataFrame(scores)

    return run_folds(folds), run_folds(week_folds).mean()

def compare(feature_sets, baseline):
    """Evaluate each feature set and compare it with the `baseline` set, using the decision rule above."""
    results = {name: evaluate(columns) for name, columns in feature_sets.items()}
    baseline_folds, baseline_weeks = results[baseline]

    rows = {}
    for name, (fold_scores, week_scores) in results.items():
        cv_change = fold_scores["macro_f1"] - baseline_folds["macro_f1"]
        noise = 2 * cv_change.std() / np.sqrt(len(cv_change))
        week_change = week_scores["macro_f1"] - baseline_weeks["macro_f1"]
        pr_auc_not_lower = (fold_scores["disrupted_pr_auc"].mean() >= baseline_folds["disrupted_pr_auc"].mean()
                            and week_scores["disrupted_pr_auc"] >= baseline_weeks["disrupted_pr_auc"])

        if name == baseline:
            verdict = ""
        elif cv_change.mean() > noise and week_change > 0 and pr_auc_not_lower:
            verdict = "better"
        elif cv_change.mean() < -noise and week_change < 0:
            verdict = "worse"
        else:
            verdict = "no difference"
        rows[name] = {
            "CV macro_f1": fold_scores["macro_f1"].mean(),
            "CV disrupted_pr_auc": fold_scores["disrupted_pr_auc"].mean(),
            "CV macro_f1 change": cv_change.mean(),
            "CV noise": noise,
            "unseen-weeks macro_f1": week_scores["macro_f1"],
            "unseen-weeks disrupted_pr_auc": week_scores["disrupted_pr_auc"],
            "unseen-weeks macro_f1 change": week_change,
            "verdict": verdict,
        }
    return pd.DataFrame(rows).T

Weeks per validation fold: [126, 124, 127, 124, 125]


### Is Each Core Feature Needed?
Remove one core feature at a time. If the scores drop clearly, the feature is pulling its weight.

In [14]:
drop_one_sets = {"core (all 4)": core_features}
for feature in core_features:
    drop_one_sets[f"without {feature}"] = [f for f in core_features if f != feature]

drop_one_table = compare(drop_one_sets, baseline="core (all 4)")
drop_one_table.round(3)

,CV macro_f1,CV disrupted_pr_auc,CV macro_f1 change,CV noise,unseen-weeks macro_f1,unseen-weeks disrupted_pr_auc,unseen-weeks macro_f1 change,verdict
core (all 4),0.538343,0.3016,0.0,0.0,0.542153,0.292521,0.0,
without geopolitical_risk_score,0.435896,0.126084,-0.102447,0.005366,0.433602,0.118405,-0.10855,worse
without port_congestion_index,0.476498,0.220006,-0.061845,0.003935,0.476459,0.221379,-0.065694,worse
without weather_disruption_score,0.476016,0.21859,-0.062327,0.006379,0.477172,0.231743,-0.064981,worse
without trade_volume_tonnes,0.536218,0.303613,-0.002124,0.003256,0.538574,0.299123,-0.003579,no difference


**3 of the 4 core features are needed.**
- Removing `geopolitical_risk_score` drops macro-F1 by about 0.10 in both checks, and removing `port_congestion_index` or `weather_disruption_score` by about 0.06. Disrupted PR-AUC falls from about 0.30 to 0.12-0.23. These are the main signals.
- Removing `trade_volume_tonnes` makes **no difference**: macro-F1 changes by about 0.002 in cross-validation (smaller than the noise) and Disrupted PR-AUC is even slightly higher without it. Its weak correlation in 02_1 doesn't add anything the other three don't already cover, so it is left out (the simpler feature set wins). It stays in `supply_chain_disruption.csv` if anyone wants to re-test it.

In [15]:
# Keep a core feature only if removing it makes the model clearly worse; if it makes no difference, the simpler set wins
selected_core = [f for f in core_features if drop_one_table.loc[f"without {f}", "verdict"] == "worse"]
print("Selected core features:", selected_core)

Selected core features: ['geopolitical_risk_score', 'port_congestion_index', 'weather_disruption_score']


### Does the Event Group Help?
Add the optional event group on top of the core features.

For comparison, three more rows:
- **core + late-2026 flag**: a temporary 0/1 column that is 1 for weeks from 2026-06-07 and says nothing else. It is a diagnostic, never a feature. Disrupted is much more common in those weeks (01), so this row shows how much a model gains just from knowing **which period** a row comes from.
- **every column except the leaks**: all numeric columns except `shipping_delay_days` and `freight_cost_usd`, including everything dropped in sections 4-6
- **every column (with freight)**: the same, plus `freight_cost_usd`, to show how much the freight leak would inflate the scores

In [16]:
every_numeric_column = [
    c for c in merged_df.columns
    if c not in id_columns + [target_column] + target_leak_columns
    and pd.api.types.is_numeric_dtype(merged_df[c])
]

# Diagnostic only: 1 for the last 30 weeks. Added to train_df for this comparison, never saved
train_df["late_2026_flag"] = (train_df["date"] >= pd.Timestamp("2026-06-07")).astype(int)

group_table = compare({
    "core": selected_core,
    "core + events": selected_core + optional_groups["events"],
    "core + late-2026 flag (diagnostic)": selected_core + ["late_2026_flag"],
    f"every column except the leaks ({len(every_numeric_column)})": every_numeric_column,
    f"every column (with freight) ({len(every_numeric_column) + 1})": every_numeric_column + ["freight_cost_usd"],
}, baseline="core")
group_table.round(3)

,CV macro_f1,CV disrupted_pr_auc,CV macro_f1 change,CV noise,unseen-weeks macro_f1,unseen-weeks disrupted_pr_auc,unseen-weeks macro_f1 change,verdict
core,0.536218,0.303613,0.0,0.0,0.538574,0.299123,0.0,
core + events,0.54539,0.308846,0.009172,0.005488,0.546074,0.299542,0.007501,better
core + late-2026 flag (diagnostic),0.542446,0.338096,0.006228,0.004089,0.548471,0.324296,0.009898,better
every column except the leaks (31),0.563552,0.315596,0.027334,0.003778,0.562411,0.313275,0.023837,better
every column (with freight) (32),0.565961,0.32235,0.029743,0.006308,0.569813,0.316539,0.031239,better


**The event group helps a little, so it is used.** Adding it improves macro-F1 by about 0.009 in cross-validation and 0.008 in the unseen-weeks check, and Disrupted PR-AUC doesn't drop. The gain is small, so 03 should re-check it with the final model.

**The late-2026 flag shows a "which period" effect.** A column that only says "this row is from the last 30 weeks" improves both checks and gives the biggest jump in Disrupted PR-AUC (about 0.30 -> 0.33). That's because Disrupted is much more common in those weeks (01). It says nothing about **why** a route is disrupted, only **when** disruptions were common.

**Adding back every dropped column scores higher (about +0.025 macro-F1), but mostly for the same reason.** Container availability and the market prices change in the last 30 weeks (section 6, and 01's commodity mismatch), so they let the model work out which period a row is from, the same way the late-2026 flag does. The route-level columns let it recognize routes. These columns are still not used:
- the team treats every route-week as independent and time as not relevant, so columns whose value comes from identifying the period go against that choice
- they explain **when** disruptions happened rather than **what causes** them, which is what the model is supposed to show

Because the split is random, the test set also contains rows from the last 30 weeks, so these columns would probably raise the test score too. That gain wouldn't come from understanding disruptions, so it is left out on purpose.

**The freight leak is visible.** Adding `freight_cost_usd` on top of everything else raises the scores again (about +0.007 macro-F1 in the unseen-weeks check). That is the piece of the delay hidden inside freight cost (section 6).

## 9. Final Feature List
The selected core features, plus the event group if it was "better" in section 8. Both choices are made by the code from the verdicts above, so re-running the notebook keeps the list consistent with the decision rule.

In [17]:
model_features = selected_core + (optional_groups["events"] if group_table.loc["core + events", "verdict"] == "better" else [])

print(f"{len(model_features)} model features:")
for feature in model_features:
    print(" ", feature)

7 model features:
  geopolitical_risk_score
  port_congestion_index
  weather_disruption_score
  origin_active_event_risk
  destination_active_event_risk
  covid_supply_shock_active
  russia_ukraine_war_active


## 10. Save the Model-Ready Data
Save to `data/preprocessed/model_ready.csv` so 03 can start from here. It keeps:
- `date`, `route_id`: to identify each row, and to join route/country details back for dashboards
- `split`: `train` or `test`, so every notebook uses the same cutoff
- `route_status`: the target
- the model features

The output format matches 01: `date` as `MM/DD/YYYY` and `route_id` as `R1` ... `R50`.

In [18]:
model_ready_df = merged_df[id_columns + [target_column] + model_features]

# Sanity checks: same rows as the merged file, nothing missing, no leak
assert len(model_ready_df) == len(merged_df)
assert model_ready_df.isna().sum().sum() == 0
assert not set(target_leak_columns) & set(model_ready_df.columns)

os.makedirs("../data/preprocessed", exist_ok=True)
model_ready_df.to_csv("../data/preprocessed/model_ready.csv", index=False, date_format="%m/%d/%Y")

print("Saved", model_ready_df.shape, "to ../data/preprocessed/model_ready.csv")
model_ready_df.head()

Saved (31300, 11) to ../data/preprocessed/model_ready.csv


,date,route_id,split,route_status,geopolitical_risk_score,port_congestion_index,weather_disruption_score,origin_active_event_risk,destination_active_event_risk,covid_supply_shock_active,russia_ukraine_war_active
0,2015-01-04,R1,train,Delayed,57.15,82.22,59.51,72.72,44.56,0,0
1,2015-01-11,R1,train,Delayed,41.47,72.01,91.27,98.86,145.96,0,0
2,2015-01-18,R1,train,Normal,79.69,44.64,46.69,142.94,145.96,0,0
3,2015-01-25,R1,train,Delayed,67.15,97.82,56.39,142.94,183.98,0,0
4,2015-02-01,R1,train,Delayed,7.59,73.21,95.92,198.03,195.50,0,0


## 11. Summary
**Train/test split:** random and stratified, 80% train / 20% test, with the same status mix in both (about 33% Normal, 64% Delayed, 3% Disrupted).

**What we decided**

| Kept / dropped | Columns | Why |
|---|---|---|
| **Kept: core (3)** | `geopolitical_risk_score`, `port_congestion_index`, `weather_disruption_score` | Removing any of them clearly hurts both checks |
| **Kept: events (4)** | `origin_active_event_risk`, `destination_active_event_risk`, `covid_supply_shock_active`, `russia_ukraine_war_active` | Small but consistent improvement in both checks |
| Not selected (1) | `trade_volume_tonnes` | Removing it made no difference |
| Dropped: target leaks (2) | `shipping_delay_days`, `freight_cost_usd` | The delay decides the status; freight cost was calculated partly from the delay |
| Dropped: route-level (24) | country, region, income group, iso3, port capacity, logistics, trade dependency, GDP, population (origin and destination), shipping method, route type, distance, transit days | Never change within a route, so they only identify the route; no signal (02_1) |
| Dropped: week-level (6) | the market prices | Same for every route in a week; they mainly identify the period |
| Dropped: copies (4) | `container_availability_index`, `carbon_emissions_tonnes`, event counts | Can be rebuilt from columns we keep; container availability only adds "which period" |
| Dropped: bookkeeping (1) | `events_warmup` | A note from 01, not a feature |

**Saved:** `data/preprocessed/model_ready.csv` with `date`, `route_id`, `split`, `route_status` and the 7 model features. `supply_chain_disruption.csv` is unchanged, so any dropped column can be brought back from it.

**Next steps for the baseline model (03)**
- Load `model_ready.csv` and set the `test` rows aside until the very end.
- Use stratified 5-fold cross-validation on the `train` rows to compare LightGBM, XGBoost and CatBoost (with class weights, no resampling).
- Tune hyperparameters and the Disrupted decision threshold inside the folds only.
- Report scores for each status (especially Disrupted recall and PR-AUC), not just accuracy.
- Extra check: also report the test score for test rows from 2026-06-07 onward (Disrupted is much more common there).
- The event features are too low in the first 52 weeks (before 2015-12-31, see 01's `events_warmup`). 03 can try training without those weeks and see if it helps.